# Tutorial for PUBMED queries + Claude 

Before we start with the code in this jupyter notebook, we first need to setup the Zotero API key and Anthropic API key in your enviroment

# Loading packages

First, we load the relevant packages

In [42]:
import argparse  # Parse command-line arguments to configure pipeline execution
import json  # Read and write structured data in JSON format
import os  # Interact with the operating system (files, paths, environment)
import xml.etree.ElementTree as ET  # Parse and navigate PubMed XML responses
from collections import defaultdict  # Dictionary with automatic default values
from datetime import datetime, timedelta  # Handle dates and date arithmetic
from time import sleep  # Pause execution to respect API rate limits
import anthropic  # Client library for interacting with Claude LLM API
import requests  # Send HTTP requests to PubMed and other REST APIs
from diskcache import Cache  # Persistent on-disk caching to avoid repeated API calls
from pyzotero import Zotero  # Interface with Zotero API for reference management

In [43]:
cache = Cache(".cache")  # Initialize persistent on-disk cache for API results and LLM outputs

zot = None  # Default to no Zotero connection unless credentials are provided
if "ZOTERO_API_KEY" in os.environ:
    zot = Zotero(
        "6377183", "group", os.environ["ZOTERO_API_KEY"]
    )  # Connect to Zotero group library using API key (cloud-based access)

# Initialize Claude client
api_key = os.environ.get("ANTHROPIC_API_KEY")  # Read Claude API key from environment variables

anthropic_client = None  # Default to no LLM client if API key is missing
if "ANTHROPIC_API_KEY" in os.environ:
    anthropic_client = anthropic.Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"]
    )  # Create Claude client for paper classification and summarization

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"  # Base URL for NCBI PubMed E-utilities API

# Helper Functions

Let's go through each function one by one:

## ***get_pmids***: querying PubMed for recent papers

In [44]:
def get_pmids(
    base_query: str,
    days_back: int = 1,
    max_results: int = 200,
):
    """
    Query PubMed using a fixed Boolean query plus a sliding publication date window.

    Parameters
    ----------
    base_query : str
        PubMed Boolean query (e.g. pharmacometrics OR clinical pharmacology)
    days_back : int
        How many days back from today to search
    max_results : int
        Maximum number of PMIDs to return

    Returns
    -------
    list[str]
        List of PubMed IDs (PMIDs)
    """

    # Get today's date in UTC (PubMed uses publication dates, not local time)
    end_date = datetime.utcnow().date()

    # Compute the start date by subtracting days_back
    start_date = end_date - timedelta(days=days_back)

    # Construct PubMed date filter syntax
    # Example:
    # "2025/01/30"[Date - Publication] : "2025/01/31"[Date - Publication]
    date_clause = (
        f'"{start_date.strftime("%Y/%m/%d")}"[Date - Publication] : '
        f'"{end_date.strftime("%Y/%m/%d")}"[Date - Publication]'
    )

    # Combine the base query with the date constraint
    # Using parentheses ensures correct Boolean precedence
    full_query = f"""
    ({base_query})
    AND
    ({date_clause})
    """

    # PubMed E-utilities endpoint for searching
    search_url = f"{BASE_URL}esearch.fcgi"

    # Parameters passed to PubMed
    search_params = {
        "db": "pubmed",        # database to search
        "term": full_query,    # search query
        "retmax": max_results, # max number of results
        "retmode": "json",     # JSON response (easier to parse)
        "usehistory": "n",     # do not store query on NCBI servers
    }

    # Execute HTTP GET request with a timeout for safety
    response = requests.get(search_url, params=search_params, timeout=30)

    # Raise an exception if PubMed returns HTTP errors (4xx / 5xx)
    response.raise_for_status()

    # Parse JSON response into Python dict
    data = response.json()

    # Safely extract list of PMIDs
    # If any key is missing, return an empty list
    return data.get("esearchresult", {}).get("idlist", [])

## ***get_article_entry***: helper to extract XML text safely

In [45]:
def get_article_entry(elem, key):
    """
    Extract text from an XML element using XPath-like syntax.

    Parameters
    ----------
    elem : xml.etree.ElementTree.Element
        Parent XML element (e.g. PubmedArticle)
    key : str
        Tag or XPath to search for

    Returns
    -------
    str or None
        Extracted text, or None if not found
    """

    # Find the first matching element
    res = elem.find(f".//{key}")

    # If found, return all text inside (handles nested tags)
    if res is not None:
        return "".join(res.itertext())

## ***query_pmid***: fetch and normalize one PubMed article

In [46]:
@cache.memoize()
def query_pmid(pmid):
    """
    Fetch full metadata for a single PubMed ID and convert it
    into a Zotero-compatible dictionary.

    Cached to avoid repeated API calls.
    """

    # PubMed efetch endpoint retrieves full records
    fetch_url = f"{BASE_URL}efetch.fcgi"

    # Parameters: database, PMID, XML output
    fetch_params = {"db": "pubmed", "id": pmid, "retmode": "xml"}

    # HTTP request to PubMed
    fetch_response = requests.get(fetch_url, params=fetch_params)

    # Parse XML response
    root = ET.fromstring(fetch_response.content)

    articles = []

    # Loop over PubmedArticle elements (normally exactly one per PMID)
    for article in root.findall(".//PubmedArticle"):

        # Extract PMID again from XML (safety)
        pmid = get_article_entry(article, "PMID")

        # Build a structured article dictionary
        article_dict = {
            "itemType": "journalArticle",  # Zotero item type
            "title": get_article_entry(article, "ArticleTitle"),
            "abstractNote": get_article_entry(article, "AbstractText"),
            "PMID": pmid,
            "date": get_article_entry(article, "PubDate"),
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "DOI": get_article_entry(article, "ArticleId[@IdType='doi']"),

            # Build list of authors
            "creators": [
                {
                    "creatorType": "author",
                    "firstName": author.findtext("ForeName"),
                    "lastName": author.findtext("LastName"),
                }
                for author in article.findall(".//AuthorList/Author")
            ],
        }

        articles.append(article_dict)

    # Sanity check: PubMed should not return multiple articles per PMID
    if len(articles) > 1:
        raise RuntimeError(f"Too many articles for given PMID ({pmid})")

    return articles[0]

## ***classify_paper***: LLM-based classification with Claude

In [47]:
FALLBACK_TAG = "Other/General"

# @cache.memoize()
def classify_paper(title, abstract):
    """
    Classify a paper into three axes (paper_type, application, methodology)
    and generate a short summary.

    Two levels of fallback:
    1. If Claude cannot run → summary indicates failure.
    2. If Claude runs but returns no tags → use 'Other/General' per axis.
    """

    if abstract is None:
        abstract = ""

    # --- Fail fast if Claude not configured ---
    if anthropic_client is None:
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Claude not configured",
        )

    prompt = f"""
You are a pharmaceutical research expert.

Classify the following paper. Assign ZERO OR MORE tags per category.
Only assign a tag if clearly supported by the title or abstract.

Return STRICT JSON only.

Paper Title:
{title}

Abstract:
{abstract[:1500]}

Tag schema:

paper_type (choose any):
- review
- tutorial
- perspective

application (choose any):
- model-informed drug development
- dose selection / optimization
- exposure–response / trial design
- precision dosing

methodology (choose any broad category):
- classical ML (supervised/unsupervised/tree-based/ensemble/feature/model selection)
- deep learning / neural networks
- Bayesian / Gaussian process
- hybrid mechanistic–ML models
- mechanism-informed machine learning
- surrogate modeling / emulation of NLME
- reinforcement learning / AI agents
- LLM / large language models
- explainable AI

Notes:
- Only assign tags clearly supported by the paper.
- If no tag applies for a category, return an empty list.

Response format:
{{
  "paper_type": [],
  "application": [],
  "methodology": [],
  "summary": "one sentence summary (max 150 chars)"
}}
"""

    try:
        # --- Call Claude API ---
        message = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )

        response_text = message.content[0].text

        # --- Try to parse JSON ---
        start_idx = response_text.find("{")
        end_idx = response_text.rfind("}") + 1
        if start_idx == -1 or end_idx <= start_idx:
            raise ValueError("No valid JSON returned from Claude")

        result = json.loads(response_text[start_idx:end_idx])

        # --- Extract summary (fallback if missing) ---
        summary = result.get(
            "summary",
            "AI/ML application in pharmacometrics or clinical pharmacology",
        )

        # --- Extract tags per axis (fallback to Other/General if empty) ---
        classification = {}
        for axis in ["paper_type", "application", "methodology"]:
            tags = result.get(axis, [])
            classification[axis] = tags if tags else [FALLBACK_TAG]

        return classification, summary

    except Exception as e:
        # --- This is a true failure: Claude call did not work ---
        print(f"Error classifying paper {title}: {e}")
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Error during classification: AI/ML application in pharmacometrics or clinical pharmacology",
        )

## ***update_readme***: generate human-readable output

In [48]:
def update_readme(articles, filename="README.md", min_citations=0):
    """
    Update README.md with papers, grouped by their multi-axis classification.

    Each paper shows:
    - Classification axes (paper_type, application, methodology)
    - Summary
    - Publication date
    - Citation count (if available)
    
    Parameters
    ----------
    articles : dict
        Dictionary of article data (must include 'classification' and 'extra')
    filename : str
        Path to the README file
    min_citations : int
        Minimum number of citations to include a paper
    """

    header = f"""# Awesome AI/ML in Pharma 🧬🤖

A curated list of recent research papers on AI/ML applications in pharmaceutical sciences, automatically updated daily.

**Last Updated**: {datetime.now().strftime('%Y-%m-%d')}

---
"""
    with open(filename, "w") as fh:
        fh.write(header)

        # Sort articles by publication date descending
        sorted_articles = sorted(
            articles.values(),
            key=lambda x: x.get("date", ""),
            reverse=True
        )

        for article in sorted_articles:
            if article.get("citations", 0) < min_citations:
                continue  # skip papers below citation threshold

            classification = article.get("classification", {})
            paper_type = ", ".join(classification.get("paper_type", ["Other/General"]))
            application = ", ".join(classification.get("application", ["Other/General"]))
            methodology = ", ".join(classification.get("methodology", ["Other/General"]))

            # Header: show multi-axis classification
            fh.write(f"\n## {paper_type} | {application} | {methodology}\n")

            fh.write(
                f"\n- **[{article['title']}]({article['url']})**"
                f"\n\t- Summary: {article.get('extra', '')}"
                f"\n\t- Published: {article.get('date', 'N/A')}"
                f"\n\t- Citations: {article.get('citations', 0)}\n"
            )


## ***main***: orchestration layer

In [49]:
def main(
    filename="all_articles.json",
    readme_path="../README.md",
    days_back=365,  # last 1 year by default
    max_results=100,
    min_citations=0,
):
    """
    Full pipeline: fetch PMIDs, retrieve details, classify papers with multi-axis tags,
    upload to Zotero, and update README with multi-axis headers.
    """

    # --- Load previously fetched papers ---
    articles = {}
    if os.path.isfile(filename):
        with open(filename, "r") as fh:
            articles = json.load(fh)

    # --- Collect PMIDs from PubMed using the search query block ---
    pmids = get_pmids(REVIEW_PUBMED_QUERY_PMX, days_back=days_back, max_results=max_results)
    print(f"🔬 Fetched {len(pmids)} PMIDs from PubMed")

    # --- Fetch new articles ---
    new_pmids = [pmid for pmid in pmids if pmid not in articles]
    print(f"📄 Fetching {len(new_pmids)} new articles from PubMed...")

    for pmid in new_pmids:
        articles[pmid] = query_pmid(pmid)
        sleep(0.5)  # rate limiting

    print(f"✅ Total articles loaded: {len(articles)}")

    # --- Determine which articles need Zotero upload ---
    pmids_to_upload = set()
    if zot is not None:
        try:
            pmids_in_zot = {x["data"]["PMID"] for x in zot.items()}
            pmids_to_upload = set(articles.keys()) - pmids_in_zot
        except Exception as e:
            print(f"Error fetching items from Zotero: {e}")

    # --- Classify papers using Claude ---
    cat2pmid = defaultdict(list)

    for pmid, article in articles.items():
        classification, summary = classify_paper(article["title"], article["abstractNote"])
        article["classification"] = classification
        article["extra"] = f"AI summary: {summary}"

        # Aggregate for README: axis:tag → PMIDs
        for axis, tags in classification.items():
            for t in tags:
                cat2pmid[f"{axis}:{t}"].append(pmid)

        # --- Prepare Zotero tags if uploading ---
        if zot is not None and pmid in pmids_to_upload:
            zot_tags = []
            for axis, tags in classification.items():
                for t in tags:
                    zot_tags.append({"tag": f"{axis}:{t}"})
            article_for_zotero = article.copy()
            article_for_zotero["tags"] = zot_tags

            try:
                zot.create_items([article_for_zotero])
                sleep(1)
            except Exception as e:
                print(f"Error uploading {pmid} to Zotero: {e}")

    # --- Save updated articles locally ---
    with open(filename, "w") as fh:
        json.dump(articles, fh, indent=1)

    print("📝 Updating README.md...")
    update_readme(articles, filename=readme_path, min_citations=min_citations)
    print("✅ Pipeline completed")

# Search terms / Categories

The pipeline will consider a main query for PUBMED API and then will classify papers based on PMX applications and the AI/ML methodology that they are considering

In [50]:
MAIN_QUERY = """(
  "machine learning"[MeSH Terms]
  OR "artificial intelligence"[MeSH Terms]
  OR "machine learning"[Title/Abstract]
  OR "artificial intelligence"[Title/Abstract]
  OR "deep learning"[Title/Abstract]
  OR "neural network*"[Title/Abstract]
  OR "random forest"[Title/Abstract]
  OR "support vector machine*"[Title/Abstract]
  OR "Gaussian process*"[Title/Abstract]
  OR "reinforcement learning"[Title/Abstract]
  OR "Bayesian machine learning"[Title/Abstract]
  OR "hybrid model*"[Title/Abstract]
  OR "mechanism-informed"[Title/Abstract]
)
AND
(
  pharmacokinetic*[Title/Abstract]
  OR pharmacodynamic*[Title/Abstract]
  OR PK/PD[Title/Abstract]
  OR "population pharmacokinetic*"[Title/Abstract]
  OR "nonlinear mixed effects"[Title/Abstract]
  OR NLME[Title/Abstract]
  OR PBPK[Title/Abstract]
  OR "physiologically based pharmacokinetic*"[Title/Abstract]
  OR QSP[Title/Abstract]
  OR "quantitative systems pharmacology"[Title/Abstract]
  OR "model-based drug development"[Title/Abstract]
  OR "model-informed drug development"[Title/Abstract]
  OR MIDD[Title/Abstract]
  OR pharmacometrics[Title/Abstract]
  OR "precision dosing"[Title/Abstract]
  OR "dose optimization"[Title/Abstract]
  OR "therapeutic drug monitoring"[Title/Abstract]
)
"""


APPLICATION_TAGS = [
  "Population PK modeling",
  "Population PD modeling",
  "PK/PD model building",
  "Dose optimization",
  "Exposure–response analysis",
  "Covariate model selection",
  "Model-informed drug development (MIDD)",
  "Clinical trial design",
  "Phase I dose escalation",
  "Precision dosing",
  "Therapeutic drug monitoring",
  "Parameter estimation",
  "Model qualification / validation",
  "Simulation and trial design",
  "PBPK modeling",
 "NLME modeling",
 "Quantitative systems pharmacology (QSP)",
 "Exposure–safety analysis",
 "Disease progression modeling",
 "Regulatory decision support",
 "Individualized therapy"
]

METHOD_TAGS = [
  "Supervised learning",
  "Unsupervised learning",
  "Deep learning",
  "Tree-based models",
  "Gaussian processes",
  "Bayesian ML",
  "Hybrid mechanistic–ML models",
  "Feature selection",
  "Model selection",
  "Surrogate modeling",
  "Emulation of NLME models",
  "Reinforcement learning",
  "Explainable AI",
    "Neural networks",
 "Ensemble learning",
 "Time-series modeling",
 "Mechanism-informed machine learning",
"LLM",
"AI Agents",
"Agentic LLM workflows"
]


In [51]:
pmids = get_pmids(
    MAIN_QUERY,
    days_back=365*25,
    max_results=5000
)
len(pmids)

/tmp/ipykernel_6101/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


1956

In [52]:
for pmid in pmids[:30]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")


- Artificial intelligence and machine learning for precision warfarin dosing: a comprehensive narrative review.
- Spatial proteomics in precision medicine: technologies, bioinformatics, and translational applications.
- Optimizing vedolizumab therapy in ulcerative colitis: A critical synthesis of trial evidence and the emerging role of artificial intelligence.
- Recent Trends in the Development and Clinical Translation of Polymer-based Targeted Therapeutic Nanoparticle.
- Stochastic Gates for Covariate Selection in Population Pharmacokinetics Modeling.
- Beyond the dose: a clearance-enabled in vitro platform for evaluating local therapies.
- The Emerging Role of Artificial Intelligence in Drug Discovery and Development: Implications for Cardiovascular Pharmacology.
- State-of-the-art 32 cm field-of-view digital PET/CT system: preliminary study for protocols optimization and DRLs update.
- Artificial intelligence and precision medicine: a pilot study predicting optimal ceftaroline dosag

# Testing the functions
 Different queries

In [9]:
ML_BLOCK = """
(
  "machine learning"[tiab] OR
  "artificial intelligence"[tiab] OR
  "deep learning"[tiab] OR
  "neural network"[tiab] OR
  "random forest"[tiab] OR
  "gradient boosting"[tiab] OR
  "XGBoost"[tiab] OR
  "support vector"[tiab]
)
"""
PMX_BLOCK = """
(
  "pharmacometrics"[tiab] OR
  "population pharmacokinetic*"[tiab] OR
  "population pharmacodynamic*"[tiab] OR
  "nonlinear mixed effect*"[tiab] OR
  "mixed effect* model*"[tiab] OR
  "NLME"[tiab] OR
  "model-based drug development"[tiab] OR
  "MIDD"[tiab] OR
  "NONMEM"[tiab] OR
  "Monolix"[tiab] OR
  "nlmixr"[tiab] OR
  "Pumas"[tiab]
)
AND
(
  "clinical pharmacology"[tiab] OR
  "drug development"[tiab] OR
  "clinical trial"[tiab] OR
  "dose finding"[tiab] OR
  "dose optimization"[tiab] OR
  "exposure response"[tiab] OR
  "model-informed drug development"[tiab] OR
  pharmacokinetic*[tiab] OR
  pharmacodynamic*[tiab] OR
  pharmacometric*[tiab]
)
"""
EXCLUDE_BLOCK = """
NOT (
  review[Publication Type] OR
  "electronic health record"[tiab] OR
  "EHR"[tiab] OR
  "radiomics"[tiab] OR
  "genome-wide"[tiab]
)
"""
BASE_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
{EXCLUDE_BLOCK}
"""



In [12]:
BROAD_QUERY = """(
  "machine learning"[MeSH Terms]
  OR "artificial intelligence"[MeSH Terms]
  OR "machine learning"[Title/Abstract]
  OR "artificial intelligence"[Title/Abstract]
  OR "deep learning"[Title/Abstract]
  OR "neural network*"[Title/Abstract]
  OR "random forest"[Title/Abstract]
  OR "support vector machine*"[Title/Abstract]
  OR "Gaussian process*"[Title/Abstract]
  OR "reinforcement learning"[Title/Abstract]
)
AND
(
  pharmacokinetic*[Title/Abstract]
  OR pharmacodynamic*[Title/Abstract]
  OR PK/PD[Title/Abstract]
  OR "population pharmacokinetic*"[Title/Abstract]
  OR "model-informed drug development"[Title/Abstract]
  OR "precision dosing"[Title/Abstract]
  OR "dose individualization"[Title/Abstract]
  OR "therapeutic drug monitoring"[Title/Abstract]
  OR PBPK[Title/Abstract]
  OR pharmacometrics[Title/Abstract]
)
"""

In [10]:
ML_BLOCK = """
(
  "machine learning"[tiab] OR
  "artificial intelligence"[tiab] OR
  "deep learning"[tiab]
)
"""
PMX_BLOCK = """
(
  "pharmacometrics"[tiab] OR
  "model-based drug development"[tiab] OR
  "MIDD"[tiab]
)
"""


REVIEW_BLOCK = """
(
  review[Publication Type] OR
  tutorial[tiab] OR
  overview[tiab] OR
  perspective[tiab] OR
  "state of the art"[tiab] OR
  guideline [tiab] OR
  "white paper" [tiab] OR
  "practical guide"[tiab]
)
"""
HUMAN_BLOCK = "humans[MeSH Terms]"

REVIEW_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
AND
{REVIEW_BLOCK}
AND
{HUMAN_BLOCK}
"""

REVIEW_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
AND
{HUMAN_BLOCK}
"""


In [13]:
pmids = get_pmids(
    REVIEW_PUBMED_QUERY_PMX,
    days_back=365*25,
    max_results=100
)
len(pmids)

/tmp/ipykernel_7285/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


61

In [14]:
for pmid in pmids[:50]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")


- Catalyzing change in MID3 through globalization, education, and innovation.
- How AI Transforms Regulatory Submission: Current Clinical Implementation and Future Prospects.
- Opportunities for AI-based Model-informed Drug Development: A Comparative Analysis of NONMEM and AI-based Models for Population Pharmacokinetic Prediction.
- AI for NONMEM Coding in Pharmacometrics Research and Education: Shortcut or Pitfall?
- Redefining Parameter Estimation and Covariate Selection via Variational Autoencoders: One Run Is All You Need.
- Artificial intelligence coupled to pharmacometrics modelling to tailor malaria and tuberculosis treatment in Africa.
- Building Hybrid Pharmacometric-Machine Learning Models in Oncology Drug Development: Current State and Recommendations.
- Mitigate Japan's Drug Loss With Model-Informed Drug Development.
- Advancing drug development with "Fit-for-Purpose" modeling informed approaches.
- Moving From Point-Based Analysis to Systems-Based Modeling: Knowledge Integ

In [ ]:
print("Found paper:", any("Adoption of Machine Learning in Pharmacometrics" in query_pmid(p)["title"] for p in pmids))


In [15]:
pmid = "36145562"

# Test each block separately
ml_hit = get_pmids(ML_BLOCK, days_back=365*5)
pmx_hit = get_pmids(PMX_BLOCK, days_back=365*5)
review_hit = get_pmids(REVIEW_BLOCK, days_back=365*5)
human_hit = get_pmids(HUMAN_BLOCK, days_back=365*5)

print(pmid in ml_hit, pmid in pmx_hit, pmid in review_hit, pmid in human_hit)


/tmp/ipykernel_7285/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


False False False False


In [16]:
REVIEW_PUBMED_QUERY_PMX = f"""
("machine learning"[tiab] OR "artificial intelligence"[tiab] OR "deep learning"[tiab])
AND
("pharmacometrics"[tiab] OR "model-based drug development"[tiab] OR "MIDD"[tiab] 
     OR "population PK"[tiab] OR "population PD"[tiab])
"""


In [17]:
pmids = get_pmids(
    REVIEW_PUBMED_QUERY_PMX,
    days_back=365*25,
    max_results=100
)
len(pmids)

/tmp/ipykernel_7285/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


100

In [18]:
for pmid in pmids[:50]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")


- Pharmacokinetics and Customized Dosing of Vancomycin in Adult Patients With Hematological Malignancies: Status, Challenges, and Opportunities.
- Mechanism-informed machine learning for individualized tacrolimus dose adjustment in the early post-kidney transplant period.
- Catalyzing change in MID3 through globalization, education, and innovation.
- Predicting prolonged dalbavancin exposure using machine learning: a validated strategy for individualized redosing.
- How AI Transforms Regulatory Submission: Current Clinical Implementation and Future Prospects.
- Uncovering Population PK Covariates from VAE-Generated Latent Spaces.
- Opportunities for AI-based Model-informed Drug Development: A Comparative Analysis of NONMEM and AI-based Models for Population Pharmacokinetic Prediction.
- AI for NONMEM Coding in Pharmacometrics Research and Education: Shortcut or Pitfall?
- Redefining Parameter Estimation and Covariate Selection via Variational Autoencoders: One Run Is All You Need.
- Ph

# Classifying the paper

In [20]:
article = query_pmid(pmids[3])
print(f"- {article['title']}")

- Predicting prolonged dalbavancin exposure using machine learning: a validated strategy for individualized redosing.


In [21]:
print(f"- {article['abstractNote']}")

- Dalbavancin is a long-acting lipoglycopeptide increasingly used off-label for complex Gram-positive infections requiring prolonged therapy. Its extended half-life enables simplified regimens, but interindividual pharmacokinetic variability and pathogen MIC heterogeneity complicate dosing. We developed and externally validated machine learning (ML) models to predict whether dalbavancin plasma concentrations remain above predefined pharmacokinetic/pharmacodynamic targets after two standard 1,500 mg doses (day 1/day 8 or day 1/day 15). Predictions were binary (adequate vs subtherapeutic concentration), directly reflecting the clinical decision to readminister a 1,500 mg dose. Models were trained on simulated PK profiles from a published population PK (popPK) model and evaluated in three independent settings: (i) simulated validation data sets from two alternative published popPK models, (ii) a real-world cohort from Limoges University Hospital (n = 31), and (iii) a secondary cohort from

In [22]:
classify_paper(article["title"], article["abstractNote"])


--- CLAUDE RAW RESPONSE ---
{
  "paper_type": [],
  "application": ["precision dosing", "dose optimization"],
  "methodology": ["machine learning", "population PK/PD", "Bayesian"],
  "summary": "Machine learning models predict dalbavancin plasma concentrations to guide individualized redosing decisions in clinical practice."
}
--- END RESPONSE ---


--- CLAUDE RAW RESPONSE ---
{
  "paper_type": [],
  "application": ["precision dosing", "dose optimization"],
  "methodology": ["machine learning", "population PK/PD", "Bayesian"],
  "summary": "Machine learning models predict dalbavancin plasma concentrations to guide individualized redosing decisions in clinical practice."
}
--- END RESPONSE ---



({'paper_type': ['Other/General'],
  'application': ['precision dosing', 'dose optimization'],
  'methodology': ['machine learning', 'population PK/PD', 'Bayesian']},
 'Machine learning models predict dalbavancin plasma concentrations to guide individualized redosing decisions in clinical practice.')

In [ ]:
print(classification)
print(summary)

In [53]:
main()

/tmp/ipykernel_6101/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


🔬 Fetched 19 PMIDs from PubMed
📄 Fetching 19 new articles from PubMed...
✅ Total articles loaded: 19
📝 Updating README.md...
✅ Pipeline completed
